# Palm Oil Grievance Classifier — Results Analysis

**Analysis-only companion** to `Twitter_Classification_Training_Validation_Test.ipynb`.

This notebook **does not train or load the model**. It reads the CSV/JSON artifacts a model run
already wrote to its `RESULTS_DIR` (the *Save Results* / *Save Test Predictions & Metrics* cells,
§11 and §15 of the training notebook) and reproduces every end-of-notebook analysis and figure —
so you can explore a run, tweak plots, and export poster figures without a GPU.

## What it expects in `RESULTS_DIR`
A folder named `{RUN_NAME}_results/` containing:

| File | Provides |
|------|----------|
| `summary.json` | thresholds, hyperparameters, `num_labels`, run name |
| `test_predictions.csv` | test probabilities, hard preds (global & per-topic), truth, text |
| `val_predictions.csv` | validation probabilities, preds, truth |
| `cv_oof_predictions.csv` | 5-fold out-of-fold probabilities + truth (the "CV / train" estimate) |
| `val_confusion_by_topic.csv`, `test_confusion_*.csv` | per-topic confusion tables (recomputed here too) |
| `train_split.csv` | *(optional)* training rows — only used for the keyword-grounded FP analysis |

## Sections
1. Imports · 2. Select the run · 3. Load the saved outputs · 4. Metrics helper
5. Train / Validation / Test performance · 6. Test-set per-topic confusion
7. Additional diagnostics · 8. Threshold-free ranking visualizations · 9. Prediction-outcome scatter

# 1. Imports

In [ ]:
import json
from pathlib import Path
from math import comb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from sklearn.metrics import f1_score, accuracy_score

pd.set_option("display.max_colwidth", None)
print("Imports ready (no torch / transformers needed).")

# 2. Select the Run to Analyze

Point `RESULTS_DIR` at the `{RUN_NAME}_results/` folder written by the training notebook. The
default path mirrors the training notebook's naming; override `RESULTS_DIR` directly if your copy
lives elsewhere (e.g. a local download).

In [ ]:
# Mount Google Drive if running on Colab (skipped automatically elsewhere).
try:
    from google.colab import drive
    drive.mount("/content/gdrive")
except Exception:
    print("Not on Colab (or Drive already mounted) — using local paths.")

# ---- Point this at the run you want to analyze ----
RUN_NAME   = "20260721_Twitter_multilabel_CVLR_non-weighted"
DRIVE_ROOT = "/content/gdrive/MyDrive/Group 3: palm oil topic classifier"
MODELS_DIR = f"{DRIVE_ROOT}/Text Classification Models/Twitter_Saved_Models"
RESULTS_DIR = Path(f"{MODELS_DIR}/{RUN_NAME}_results")

# Or override directly, e.g. a local copy of the results folder:
# RESULTS_DIR = Path("./20260721_Twitter_multilabel_CVLR_non-weighted_results")

assert RESULTS_DIR.exists(), f"RESULTS_DIR not found: {RESULTS_DIR}"
print("Analyzing:", RESULTS_DIR)
print("Files present:")
for p in sorted(RESULTS_DIR.iterdir()):
    print(f"  {p.name}")

# 3. Load the Saved Run Outputs

Reconstruct every variable the analysis cells expect — `prediction_df`, `test_df`, the
`*_probs` / `*_labels` / `*_preds` arrays, the OOF matrices, and the selected thresholds — straight
from the saved files. Nothing is recomputed from the model.

In [ ]:
# --- summary + selected thresholds ---
with open(RESULTS_DIR / "summary.json") as f:
    summary = json.load(f)

NUM_LABELS = int(summary["num_labels"])
TOPIC_COLS = [f"Topic_{i}" for i in range(NUM_LABELS)]
prob_cols  = [f"Topic_{i}_Prob" for i in range(NUM_LABELS)]
RUN_NAME   = summary["run_name"]

BEST_THRESHOLD_GLOBAL     = float(summary["selected"]["best_threshold_global"])
BEST_THRESHOLDS_PER_TOPIC = np.array(summary["selected"]["best_thresholds_per_topic"], dtype=np.float32)


def _mat(df, prefix):
    """Stack the NUM_LABELS per-topic columns sharing a prefix into an (n × K) array."""
    return df[[f"{prefix}{i}" for i in range(NUM_LABELS)]].values


# --- TEST predictions ---
_test = pd.read_csv(RESULTS_DIR / "test_predictions.csv")
test_pks      = _test["pk"].values
test_probs    = _mat(_test, "prob_Topic_").astype(np.float32)
test_labels   = _mat(_test, "true_Topic_").astype(int)
test_preds    = _mat(_test, "pred_global_Topic_").astype(int)      # hard preds @ global threshold
test_preds_pt = _mat(_test, "pred_pertopic_Topic_").astype(int)   # hard preds @ per-topic thresholds

# prediction_df / test_df in the exact shape the analysis cells expect
prediction_df = pd.DataFrame({"pk": test_pks, "Text": _test["Text"].values})
for i in range(NUM_LABELS):
    prediction_df[f"Topic_{i}_Prob"] = test_probs[:, i]
test_df = pd.DataFrame({"pk": test_pks, "Text": _test["Text"].values})
for i in range(NUM_LABELS):
    test_df[f"Topic_{i}"] = test_labels[:, i]

prob_mat = prediction_df[prob_cols].values
true_mat = test_df.set_index("pk").loc[prediction_df["pk"], TOPIC_COLS].values.astype(int)

# --- VALIDATION predictions ---
_val = pd.read_csv(RESULTS_DIR / "val_predictions.csv")
val_probs  = _mat(_val, "prob_Topic_").astype(np.float32)
val_preds  = _mat(_val, "pred_Topic_").astype(int)
val_labels = _mat(_val, "true_Topic_").astype(int)

# --- CV out-of-fold predictions (the "train / CV" estimate) ---
_oof = pd.read_csv(RESULTS_DIR / "cv_oof_predictions.csv")
oof_probs     = _mat(_oof, "prob_Topic_").astype(np.float32)
y_train_multi = _mat(_oof, "true_Topic_").astype(int)

# --- optional: training rows, only for the keyword-grounded FP analysis (§7) ---
_train_split = RESULTS_DIR / "train_split.csv"
dominant_topic_df = pd.read_csv(_train_split) if _train_split.exists() else None

print(f"Loaded run: {RUN_NAME}")
print(f"  test : {len(test_labels):>4} docs")
print(f"  val  : {len(val_labels):>4} docs")
print(f"  CV   : {len(y_train_multi):>4} training rows (out-of-fold)")
print(f"  global threshold      = {BEST_THRESHOLD_GLOBAL}")
print(f"  per-topic thresholds  = {BEST_THRESHOLDS_PER_TOPIC.tolist()}")
print(f"  keyword FP analysis   = {'available' if dominant_topic_df is not None else 'skipped (no train_split.csv)'}")

# 4. Metrics Helper — per-topic confusion & rates

In [ ]:
def per_topic_confusion(y_true, y_prob, thresholds, topic_names=None):
    """Per-topic TP/FP/FN/TN and rates (sensitivity/recall, specificity,
    precision/PPV, NPV, F1, accuracy) with macro/micro summaries.

    thresholds : scalar (same threshold for every topic) or length-K array.
    Returns (per_topic_df, macro_series, micro_dict).
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    K = y_true.shape[1]
    thr = (np.full(K, float(thresholds)) if np.isscalar(thresholds)
           else np.asarray(thresholds, dtype=float))
    topic_names = topic_names or [f"Topic_{k}" for k in range(K)]

    def sd(a, b):
        return a / b if b else 0.0

    rows = []
    for k in range(K):
        preds = (y_prob[:, k] >= thr[k]).astype(int)
        tr    = y_true[:, k]
        tp = int(((tr == 1) & (preds == 1)).sum())
        tn = int(((tr == 0) & (preds == 0)).sum())
        fp = int(((tr == 0) & (preds == 1)).sum())
        fn = int(((tr == 1) & (preds == 0)).sum())
        n  = tp + tn + fp + fn
        sens = sd(tp, tp + fn)      # sensitivity / recall / TPR
        spec = sd(tn, tn + fp)      # specificity / TNR
        prec = sd(tp, tp + fp)      # precision / PPV
        npv  = sd(tn, tn + fn)
        rows.append({
            "topic": topic_names[k], "thr": round(float(thr[k]), 2),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn, "support": tp + fn,
            "TP_rate": round(sd(tp, n), 3), "FP_rate": round(sd(fp, n), 3),
            "FN_rate": round(sd(fn, n), 3), "TN_rate": round(sd(tn, n), 3),
            "sensitivity": round(sens, 3), "specificity": round(spec, 3),
            "precision": round(prec, 3), "npv": round(npv, 3),
            "f1": round(sd(2 * prec * sens, prec + sens), 3),
            "accuracy": round(sd(tp + tn, n), 3),
        })
    df = pd.DataFrame(rows)

    macro = df[["sensitivity", "specificity", "precision", "npv", "f1", "accuracy"]].mean().round(3)
    T = df[["TP", "FP", "FN", "TN"]].sum()
    total = int(T["TP"] + T["TN"] + T["FP"] + T["FN"])
    mp = sd(T["TP"], T["TP"] + T["FP"])
    ms = sd(T["TP"], T["TP"] + T["FN"])
    micro = {
        "sensitivity": round(ms, 3),
        "specificity": round(sd(T["TN"], T["TN"] + T["FP"]), 3),
        "precision":   round(mp, 3),
        "f1":          round(sd(2 * mp * ms, mp + ms), 3),
        "accuracy":    round(sd(T["TP"] + T["TN"], total), 3),
    }
    return df, macro, micro


# Per-topic confusion & rates at the CV-selected GLOBAL threshold, for val and test.
val_conf_df,  val_macro,  val_micro  = per_topic_confusion(
    val_labels,  val_probs,  BEST_THRESHOLD_GLOBAL, topic_names=TOPIC_COLS)
test_conf_global, test_macro_g, test_micro_g = per_topic_confusion(
    test_labels, test_probs, BEST_THRESHOLD_GLOBAL, topic_names=TOPIC_COLS)

print(f"VALIDATION set  @ threshold {BEST_THRESHOLD_GLOBAL}")
print(val_conf_df.to_string(index=False))
print("  macro:", "  ".join(f"{k}={v}" for k, v in val_macro.items()))
print("  micro:", "  ".join(f"{k}={v}" for k, v in val_micro.items()))
print("\n" + "-" * 74 + "\n")
print(f"TEST set  @ threshold {BEST_THRESHOLD_GLOBAL}")
print(test_conf_global.to_string(index=False))
print("  macro:", "  ".join(f"{k}={v}" for k, v in test_macro_g.items()))
print("  micro:", "  ".join(f"{k}={v}" for k, v in test_micro_g.items()))

# 5. Train / Validation / Test Performance

Macro-F1 stat tiles across the three splits, then the per-topic + overall validation-vs-test dumbbell.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import f1_score

# --- the three macro-F1 numbers, all at the SAME global threshold (fair comparison) ---
t = BEST_THRESHOLD_GLOBAL
cv_macro   = f1_score(y_train_multi, (oof_probs  >= t).astype(int), average="macro", zero_division=0)
val_macro  = f1_score(val_labels,    (val_probs  >= t).astype(int), average="macro", zero_division=0)
test_macro = f1_score(test_labels,   (test_probs >= t).astype(int), average="macro", zero_division=0)
# (or just hardcode: cv_macro, val_macro, test_macro = 0.75, 0.77, 0.49)

tiles = [
    {"label": "CV (out-of-fold)", "value": cv_macro,   "accent": "#009E73",
     "ctx": f"5-fold CV on training set · n={len(y_train_multi)}"},
    {"label": "Validation",       "value": val_macro,  "accent": "#0072B2",
     "ctx": f"held-out · same corpus · n={len(val_labels)}"},
    {"label": "Test",             "value": test_macro, "accent": "#D55E00",
     "ctx": f"held-out · different corpus · n={len(test_labels)}"},
]

fig, axes = plt.subplots(1, 3, figsize=(11, 2.6))
fig.suptitle(f"Macro-F1 across splits   (Classification Threshold = {t})",
             fontsize=20, fontweight="bold", y=0.9)
for ax, tile in zip(axes, tiles):
    ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.add_patch(plt.Rectangle((0.02, 0.05), 0.96, 0.90, facecolor="#F5F5F5",
                               edgecolor="#DDDDDD", lw=1))                 # card
    ax.add_patch(plt.Rectangle((0.02, 0.86), 0.96, 0.09, facecolor=tile["accent"],
                               edgecolor="none"))                          # top accent strip
    ax.text(0.5, 0.70, tile["label"].upper(), ha="center", va="center",
            fontsize=12, fontweight="bold", color="#333333")
    ax.text(0.5, 0.43, f"{tile['value']:.2f}", ha="center", va="center",
            fontsize=40, fontweight="bold", color="#1A1A1A")
    ax.text(0.5, 0.16, tile["ctx"], ha="center", va="center",
            fontsize=9.5, color="#666666")
plt.tight_layout()
fig.savefig("macro_f1_tiles.pdf", bbox_inches="tight")

In [ ]:
import numpy as np, matplotlib.pyplot as plt

# per-topic F1 + support from §11 (val) and §13/§14 (test) confusion tables
val_f1  = val_conf_df.set_index("topic")["f1"]
test_f1 = test_conf_global.set_index("topic")["f1"]      # per_topic_confusion(test_labels, test_probs, BEST_THRESHOLD_GLOBAL)
support = test_conf_global.set_index("topic")["support"]

NAMES = {"Topic_0": "Failed Compensation / Land Conflicts", "Topic_1": "Environmental Impact",
         "Topic_2": "Administrative Violations", "Topic_3": "Deforestation",
         "Topic_4": "Labor & Human Rights", "Topic_5": "Illegal / Contaminated Fruit Bunches"}

order = test_f1.sort_values().index                       # weak topics at the bottom
labels = [f"{NAMES[t]}  (n={int(support[t])})" for t in order] + ["Overall (macro-F1)"]
vals   = [val_f1[t]  for t in order] + [val_f1.mean()]
tests  = [test_f1[t] for t in order] + [test_f1.mean()]

C_VAL, C_TEST = "#0072B2", "#D55E00"                       # Okabe-Ito blue / vermillion (CVD-safe)
y = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(9.5, 5.5))
for yi, v, t in zip(y, vals, tests):
    ax.plot([v, t], [yi, yi], color="#BBBBBB", lw=2.5, zorder=1)
ax.scatter(vals,  y, s=120, color=C_VAL,  zorder=3, label="Validation (same distribution)")
ax.scatter(tests, y, s=120, color=C_TEST, zorder=3, label="Test (held-out, different corpus)")
for yi, v, t in zip(y, vals, tests):
    ax.annotate(f"{v:.2f}", (v, yi), xytext=(0, 9),  textcoords="offset points",
                ha="center", color=C_VAL,  fontsize=9, fontweight="bold")
    ax.annotate(f"{t:.2f}", (t, yi), xytext=(0, -15), textcoords="offset points",
                ha="center", color=C_TEST, fontsize=9, fontweight="bold")
ax.axhline(y[-1] - 0.5, color="black", lw=0.8, ls=":")    # set the Overall row apart
ax.set_yticks(y); ax.set_yticklabels(labels)
ax.set_xlim(0, 1.0); ax.set_xlabel("F1  (decision threshold = 0.35)")
ax.set_title("Per-topic and overall F1: validation vs. held-out test", fontweight="bold", fontsize=14)
ax.spines[["top", "right"]].set_visible(False)
ax.xaxis.grid(True, ls=":", alpha=0.4); ax.set_axisbelow(True)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
fig.savefig("val_vs_test_f1.pdf", bbox_inches="tight")

# 6. Test-Set Per-Topic Confusion (global vs. per-topic thresholds)

In [ ]:
# Per-topic confusion & rates on the TEST set, for both threshold strategies.
test_true_mat = test_df.set_index("pk").loc[prediction_df["pk"], TOPIC_COLS].values.astype(int)
test_prob_mat = prediction_df[prob_cols].values

for name, thr in [(f"GLOBAL threshold = {BEST_THRESHOLD_GLOBAL}", BEST_THRESHOLD_GLOBAL),
                  ("PER-TOPIC thresholds", BEST_THRESHOLDS_PER_TOPIC)]:
    df, macro, micro = per_topic_confusion(test_true_mat, test_prob_mat, thr, topic_names=TOPIC_COLS)
    print("=" * 74)
    print(name)
    print("=" * 74)
    print(df.to_string(index=False))
    print("\n  Macro:", "  ".join(f"{k}={v}" for k, v in macro.items()))
    print("  Micro:", "  ".join(f"{k}={v}" for k, v in micro.items()), "\n")

# Side-by-side comparison of the headline numbers (per_topic - global).
g_df, g_macro, g_micro = per_topic_confusion(test_true_mat, test_prob_mat, BEST_THRESHOLD_GLOBAL, TOPIC_COLS)
p_df, p_macro, p_micro = per_topic_confusion(test_true_mat, test_prob_mat, BEST_THRESHOLDS_PER_TOPIC, TOPIC_COLS)
comparison = pd.DataFrame({
    "metric": ["macro_sensitivity", "macro_specificity", "macro_precision", "macro_f1",
               "micro_sensitivity", "micro_precision", "micro_f1"],
    "global": [g_macro["sensitivity"], g_macro["specificity"], g_macro["precision"], g_macro["f1"],
               g_micro["sensitivity"], g_micro["precision"], g_micro["f1"]],
    "per_topic": [p_macro["sensitivity"], p_macro["specificity"], p_macro["precision"], p_macro["f1"],
                  p_micro["sensitivity"], p_micro["precision"], p_micro["f1"]],
})
comparison["delta"] = (comparison["per_topic"] - comparison["global"]).round(3)
print("=" * 74)
print("COMPARISON (per_topic - global)")
print("=" * 74)
print(comparison.to_string(index=False))

# 7. Additional Diagnostics

Keyword-grounded false-positive audit, and threshold-free top-1 / top-2 hit rates.

In [ ]:
import numpy as np, pandas as pd

# topic -> its BERTopic keyword string (from the training-derived column)
if (dominant_topic_df is not None
        and {"Topic_Keywords", "Dominant_Topic"}.issubset(dominant_topic_df.columns)):
    topic_keywords = (dominant_topic_df.dropna(subset=["Topic_Keywords"])
                      .groupby("Dominant_Topic")["Topic_Keywords"].first().to_dict())
else:
    topic_keywords = {}
    print("Note: Topic_Keywords/Dominant_Topic unavailable — keyword-hit columns will be empty.\n")
pk_to_text = prediction_df.set_index("pk")["Text"].to_dict()

try:
    thresholds = np.asarray(BEST_THRESHOLDS_PER_TOPIC, dtype=float)
except NameError:
    thresholds = np.full(len(prob_cols), BEST_THRESHOLD_GLOBAL, dtype=float)

rows = []
for i, prob_col in enumerate(prob_cols):
    probs  = prediction_df[prob_col].values
    truths = test_df.set_index("pk").loc[prediction_df["pk"], f"Topic_{i}"].values.astype(int)
    t      = thresholds[i]
    preds  = (probs >= t).astype(int)

    kws = [k.strip().lower() for k in str(topic_keywords.get(i, "")).split(",") if k.strip()]

    # BERTopic soft membership for this topic, if the test file carries it
    perc_col = f"Topic_{i}_Perc"
    has_perc = perc_col in test_df.columns
    perc = (test_df.set_index("pk").loc[prediction_df["pk"], perc_col].values
            if has_perc else np.full(len(probs), np.nan))

    for pk, p, tr, pr, pc in zip(prediction_df["pk"].values, probs, truths, preds, perc):
        if not (tr == 0 and pr == 1):     # keep only FPs
            continue
        text = str(pk_to_text.get(pk, "")).lower()
        matched = [k for k in kws if k in text]
        rows.append({
            "pk": pk, "topic": f"Topic_{i}",
            "bert_prob": round(float(p), 3),
            "margin_over_thr": round(float(p - t), 3),
            "bertopic_membership": round(float(pc), 3) if has_perc else None,
            "n_keyword_hits": len(matched),
            "matched_keywords": ", ".join(matched),
            "text": pk_to_text.get(pk, ""),
        })

fp_df = pd.DataFrame(rows).sort_values("bert_prob", ascending=False).reset_index(drop=True)
print(f"{len(fp_df)} false positives total\n")

# --- Signal 1: how confident is BERT on its FPs? ---
print("BERT confidence on FPs:")
print(fp_df["bert_prob"].describe()[["mean", "25%", "50%", "75%", "max"]])

# --- Signal 2: do FP docs contain the topic's own keywords? ---
frac_kw = (fp_df["n_keyword_hits"] > 0).mean()
print(f"\nFraction of FPs whose text contains >=1 of the topic's keywords: {frac_kw:.0%}")

# --- Signal 3: were these near-misses in BERTopic's own soft membership? ---
if fp_df["bertopic_membership"].notna().any():
    print("\nBERTopic membership on FP cells (if high, BERTopic 'almost' assigned it):")
    print(fp_df["bertopic_membership"].describe()[["mean", "50%", "max"]])

fp_df.head(20)

In [ ]:
import numpy as np
import pandas as pd

# True label matrix (docs x topics), aligned to prediction_df's pk order
true_mat = test_df.set_index("pk").loc[prediction_df["pk"], [f"Topic_{i}" for i in range(len(prob_cols))]].values.astype(int)
prob_mat = prediction_df[prob_cols].values                      # (docs x topics)

top1_pred = prob_mat.argmax(axis=1)                             # model's single most-probable topic per doc
n_true    = true_mat.sum(axis=1)                                # how many true topics each doc has

# A "hit" = the model's top topic is one of the document's true topics
hit = np.array([true_mat[d, top1_pred[d]] == 1 for d in range(len(top1_pred))])

# Documents with no hand-labeled topic at all can't be scored — track separately
has_label = n_true > 0
hit_rate_all   = hit.mean()
hit_rate_valid = hit[has_label].mean() if has_label.any() else float("nan")

print(f"Documents: {len(top1_pred)}  |  with >=1 hand label: {has_label.sum()}  |  unlabeled: {(~has_label).sum()}")
print(f"\nTop-1 hit rate (all docs):                {hit_rate_all:.3f}")
print(f"Top-1 hit rate (only docs with a label):  {hit_rate_valid:.3f}")
print(f"  -> In {hit[has_label].sum()}/{has_label.sum()} labeled docs, the model's most-probable "
      f"topic was one the annotators assigned.")

In [ ]:
# --- Top-2 hit: is EITHER of the model's two most-probable topics a true label? ---
top2_pred = np.argsort(-prob_mat, axis=1)[:, :2]
hit_top2 = np.array([true_mat[d, top2_pred[d]].max() == 1 for d in range(len(top2_pred))])
print(f"\nTop-2 hit rate (labeled docs): {hit_top2[has_label].mean():.3f}")

# --- Per-topic: when the model's top pick is Topic_i, how often is it correct? ---
rows = []
for i in range(len(prob_cols)):
    sel = top1_pred == i
    n_sel = sel.sum()
    correct = hit[sel].sum()
    rows.append({
        "top_topic": f"Topic_{i}",
        "times_top_pick": int(n_sel),
        "correct": int(correct),
        "precision_of_top_pick": round(correct / n_sel, 3) if n_sel else 0.0,
    })
print("\nWhen each topic is the model's #1 pick, how often is it a true label:")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
counts = pd.Series(top1_pred).value_counts().sort_index()
print(counts.rename(lambda i: f"Topic_{i}"))
print("sum of n =", int(counts.sum()), " | n_docs =", len(top1_pred))   # these must match

# 8. Threshold-Free Ranking Visualizations

Top-k hit-rate curve, per-topic top-pick precision (Wilson CIs), and per-topic detection@k facets.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from math import comb

prob_cols = [f"Topic_{i}_Prob" for i in range(NUM_LABELS)]
K = NUM_LABELS

# TEST-set probabilities (n_docs × K) and human labels, aligned to prediction_df's pk order
prob_mat = prediction_df[prob_cols].values
true_mat = (test_df.set_index("pk")
            .loc[prediction_df["pk"], [f"Topic_{i}" for i in range(K)]]
            .values.astype(int))

TOPIC_LABELS = [
    "Failed Compensation\n/ Land Conflicts", "Environmental Impact",
    "Administrative Violations", "Deforestation",
    "Labor & Human Rights", "Illegal / Contaminated\nFruit Bunches",
]

# Poster-friendly type sizes (viewers read from ~1 m away)
plt.rcParams.update({"font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13,
                     "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 12})

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from math import comb


# --- score only docs that have at least one human label ---
mask = true_mat.sum(axis=1) > 0
P, Y = prob_mat[mask], true_mat[mask]
n, K = P.shape
order = np.argsort(-P, axis=1)                 # topics ranked high->low prob, per doc

# hit@k: did the model surface >=1 true topic within its top k?
hit_at_k = [np.mean([Y[d, order[d, :k]].max() == 1 for d in range(n)]) for k in range(1, K + 1)]

# chance: random ranking. P(>=1 true in a random size-k set) = 1 - C(K-t, k)/C(K, k)
t_doc  = Y.sum(axis=1)
chance = [np.mean([1 - comb(K - t, k) / comb(K, k) for t in t_doc]) for k in range(1, K + 1)]


ks = np.arange(1, K + 1)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(ks, hit_at_k, "-o", color="#0072B2", lw=2.5, ms=9, label="Model", zorder=3)
ax.plot(ks, chance,   "--o", color="#999999", lw=2,   ms=6, label="Random ranking", zorder=2)
for k in (1, 2, 3, 4, 5, 6):                               # direct-label the headline points
    ax.annotate(f"{hit_at_k[k-1]:.0%}", (k, hit_at_k[k-1]),
                textcoords="offset points", xytext=(0, 12),
                ha="center", fontsize=12, fontweight="bold", color="#0072B2")
ax.set_xlabel("k  (top-k most probable topics considered)")
ax.set_ylabel("Docs with ≥1 correct topic in top-k")
ax.set_title("Top-k topic hit rate vs. chance", fontweight="bold", fontsize=14)
ax.set_xticks(ks); ax.set_ylim(0, 1.02)
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, ls=":", alpha=0.4); ax.set_axisbelow(True)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()

fig.savefig("topk_hit_rate.pdf", bbox_inches="tight")     # vector: crisp at any blow-up size
# raster fallback if your poster template needs an image:
# fig.savefig("topk_hit_rate.png", dpi=300, bbox_inches="tight")


In [ ]:
def wilson_ci(c, n, z=1.96):
    """Point estimate + Wilson score interval for c successes in n trials."""
    if n == 0:
        return np.nan, np.nan, np.nan
    p = c / n
    d = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / d
    half = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / d
    return p, max(0.0, center - half), min(1.0, center + half)

top1 = prob_mat.argmax(axis=1)
rows = []
for i in range(K):
    sel = top1 == i
    ni, correct = int(sel.sum()), int(true_mat[sel, i].sum())
    p, lo, hi = wilson_ci(correct, ni)
    rows.append({"topic": TOPIC_LABELS[i].replace("\n", " "), "n": ni, "p": p, "lo": lo, "hi": hi})

prec_df = pd.DataFrame(rows).dropna(subset=["p"]).sort_values("p").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
y = np.arange(len(prec_df))
xerr = np.vstack([prec_df["p"] - prec_df["lo"], prec_df["hi"] - prec_df["p"]])
ax.barh(y, prec_df["p"], color="#0072B2", height=0.62, zorder=2)
ax.errorbar(prec_df["p"], y, xerr=xerr, fmt="none", ecolor="black",
            elinewidth=1.3, capsize=4, zorder=3)
for yi, r in zip(y, prec_df.itertuples()):
    ax.text(min(r.hi + 0.02, 0.99), yi, f"{r.p:.0%}", va="center", ha="left", fontsize=11)
ax.set_yticks(y)
ax.set_yticklabels([f"{t}  (n={n})" for t, n in zip(prec_df["topic"], prec_df["n"])])
ax.set_xlim(0, 1.08)
ax.set_xlabel("Precision when this topic is the model's #1 pick")
ax.set_title("How reliable is the model's top pick, by topic?", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.xaxis.grid(True, ls=":", alpha=0.4); ax.set_axisbelow(True)
plt.tight_layout()

In [ ]:
order = (-prob_mat).argsort(axis=1)
ranks = order.argsort(axis=1)          # ranks[d, i] = rank of topic i in doc d (0 = highest prob)

ks = np.arange(1, K + 1)
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=True)
for i, ax in enumerate(axes.flat):
    pos = true_mat[:, i] == 1          # docs where Topic_i is truly present
    ni = int(pos.sum())
    detect = [np.mean(ranks[pos, i] < k) if ni else np.nan for k in ks]
    ax.plot(ks, ks / K, "--", color="#999999", lw=1.8, zorder=2, label="Chance")
    ax.plot(ks, detect, "-o", color="#0072B2", lw=2.2, ms=7, zorder=3, label="Model")
    for k in (1, 2, 3, 4, 5, 6):
        if ni:
            ax.annotate(f"{detect[k-1]:.0%}", (k, detect[k-1]), textcoords="offset points",
                        xytext=(0, 9), ha="center", fontsize=10, fontweight="bold", color="#0072B2")
    ax.set_title(f"{TOPIC_LABELS[i]}  (n={ni})", fontsize=11, fontweight="bold")
    ax.set_xticks(ks); ax.set_ylim(0, 1.05)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.grid(True, ls=":", alpha=0.4); ax.set_axisbelow(True)
    if i % 3 == 0: ax.set_ylabel("Detected in top-k")
    if i // 3 == 1: ax.set_xlabel("k")
axes.flat[0].legend(frameon=False, loc="lower right", fontsize=10)
fig.suptitle("Per-topic detection rate vs. k  —  when a topic is truly present, is it ranked in the top k?",
             fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])

# 9. Prediction Outcome Scatter

Every point is one (document, topic) pair, colored by confusion-matrix cell at the chosen threshold.

In [ ]:
COLOR_TP = "#FF9D00"
COLOR_TN = "#00A2F3"
COLOR_FP = "#4B2362"
COLOR_FN = "#CE4763"

points = []
for i, prob_col in enumerate(prob_cols):
    probs = prediction_df[prob_col].values
    truths = test_df.set_index("pk").loc[prediction_df["pk"], f"Topic_{i}"].values.astype(int)
    preds = (probs >= BEST_THRESHOLD_GLOBAL).astype(int)
    for pk, p, t, pr in zip(prediction_df["pk"].values, probs, truths, preds):
        if   t == 1 and pr == 1: c = COLOR_TP
        elif t == 0 and pr == 0: c = COLOR_TN
        elif t == 0 and pr == 1: c = COLOR_FP
        else:                    c = COLOR_FN
        points.append((pk, p, c))

fig, ax = plt.subplots(figsize=(14, 6))
for pk, p, c in points:
    ax.scatter(pk, p, color=c, alpha=0.7, s=20)
ax.axhline(BEST_THRESHOLD_GLOBAL, color="grey", linestyle="dashed",
           linewidth=2, label=f"threshold = {BEST_THRESHOLD_GLOBAL}")
ax.set_title("Predicted probability per (document, topic)")
ax.set_xlabel("Document ID (pk)")
ax.set_ylabel("Predicted probability")
ax.set_ylim(0, 1)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(handles=[
    Line2D([0], [0], marker="o", color="w", label="True Positive",  markerfacecolor=COLOR_TP, markersize=8),
    Line2D([0], [0], marker="o", color="w", label="True Negative",  markerfacecolor=COLOR_TN, markersize=8),
    Line2D([0], [0], marker="o", color="w", label="False Positive", markerfacecolor=COLOR_FP, markersize=8),
    Line2D([0], [0], marker="o", color="w", label="False Negative", markerfacecolor=COLOR_FN, markersize=8),
    Line2D([0], [0], color="grey", linestyle="dashed", label=f"threshold = {BEST_THRESHOLD_GLOBAL}"),
], title="Outcome")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

COLOR_TP = "#009E73"   # green  — correct positive
COLOR_FN = "#D55E00"   # vermillion — missed positive (bad)
COLOR_FP = "#CC79A7"   # purple — false alarm
COLOR_TN = "#BBBBBB"   # grey   — correct negative (de-emphasized; usually the majority)

TOPIC_LABELS = [
    "Failed Compensation\n/ Land Conflicts",
    "Environmental\nImpact",
    "Administrative\nViolations",
    "Deforestation",
    "Labor & Human\nRights",
    "Illegal / Contaminated\nFruit Bunches",
]

# Use per-topic thresholds if you have them, else fall back to the global one
try:
    thresholds = np.asarray(BEST_THRESHOLDS_PER_TOPIC, dtype=float)
except NameError:
    thresholds = np.full(len(prob_cols), BEST_THRESHOLD_GLOBAL, dtype=float)

rng = np.random.default_rng(42)          # reproducible jitter
fig, ax = plt.subplots(figsize=(13, 6.5))

legend_counts = {"TP": 0, "FN": 0, "FP": 0, "TN": 0}

for i, prob_col in enumerate(prob_cols):
    probs  = prediction_df[prob_col].values
    truths = test_df.set_index("pk").loc[prediction_df["pk"], f"Topic_{i}"].values.astype(int)
    t      = thresholds[i]
    preds  = (probs >= t).astype(int)

    for p, tr, pr in zip(probs, truths, preds):
        if   tr == 1 and pr == 1: c, key = COLOR_TP, "TP"
        elif tr == 1 and pr == 0: c, key = COLOR_FN, "FN"
        elif tr == 0 and pr == 1: c, key = COLOR_FP, "FP"
        else:                     c, key = COLOR_TN, "TN"
        x = i + rng.uniform(-0.28, 0.28)                       # horizontal jitter
        # de-emphasize the (usually dominant) true negatives
        ax.scatter(x, p, color=c, s=(14 if key == "TN" else 42),
                   alpha=(0.35 if key == "TN" else 0.9),
                   edgecolor="none" if key == "TN" else "black", linewidth=0.4, zorder=2)
        legend_counts[key] += 1

    # per-topic threshold segment
    ax.plot([i - 0.35, i + 0.35], [t, t], color="black", linestyle="--", linewidth=1.6, zorder=3)
    ax.text(i + 0.37, t, f"{t:.2f}", va="center", ha="left", fontsize=8, color="black")

ax.set_xticks(range(len(prob_cols)))
ax.set_xticklabels(TOPIC_LABELS, fontsize=10)
ax.set_ylabel("Predicted probability", fontweight="bold")
ax.set_ylim(-0.02, 1.02)
ax.set_title("Predicted Probabilities and Decision Thresholds by Topic",
             fontsize=15, fontweight="bold", pad=12)
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, linestyle=":", alpha=0.4)
ax.set_axisbelow(True)

ax.legend(handles=[
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_TP, markersize=9,
           label=f"True Positive ({legend_counts['TP']})"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_FN, markersize=9,
           label=f"False Negative ({legend_counts['FN']})"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_FP, markersize=9,
           label=f"False Positive ({legend_counts['FP']})"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_TN, markersize=7,
           label=f"True Negative ({legend_counts['TN']})"),
    Line2D([0],[0], color="black", linestyle="--", label="Per-topic threshold"),
], title="Outcome", loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.metrics import f1_score

COLOR_TP = "#009E73"   # correct positive
COLOR_FN = "#D55E00"   # missed positive (bad)
COLOR_FP = "#CC79A7"   # false alarm
COLOR_TN = "#BBBBBB"   # correct negative (de-emphasized)

TOPIC_LABELS = [
    "Failed Compensation / Land Conflicts",
    "Environmental Impact",
    "Administrative Violations",
    "Deforestation",
    "Labor & Human Rights",
    "Illegal / Contaminated Fruit Bunches",
]

# per-topic thresholds if available, else global
try:
    thresholds = np.asarray(BEST_THRESHOLDS_PER_TOPIC, dtype=float)
except NameError:
    thresholds = np.full(len(prob_cols), BEST_THRESHOLD_GLOBAL, dtype=float)

rng = np.random.default_rng(42)
n = len(prob_cols)
n_cols = 3
n_rows = int(np.ceil(n / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 8.5), sharey=True)
axes = axes.flatten()

for i, prob_col in enumerate(prob_cols):
    ax = axes[i]
    probs  = prediction_df[prob_col].values
    truths = test_df.set_index("pk").loc[prediction_df["pk"], f"Topic_{i}"].values.astype(int)
    t      = thresholds[i]
    preds  = (probs >= t).astype(int)

    # split into the two truth columns
    for truth_val, xbase, pos_color, neg_color in [
        (0, 0, COLOR_FP, COLOR_TN),   # negatives: above thr = FP, below = TN
        (1, 1, COLOR_TP, COLOR_FN),   # positives: above thr = TP, below = FN
    ]:
        mask = truths == truth_val
        pv   = probs[mask]
        for p in pv:
            correct_side = (p >= t) == (truth_val == 1)
            c = (pos_color if p >= t else neg_color)
            x = xbase + rng.uniform(-0.18, 0.18)
            ax.scatter(x, p, color=c,
                       s=(16 if (truth_val == 0 and p < t) else 46),   # shrink TNs
                       alpha=(0.4 if (truth_val == 0 and p < t) else 0.9),
                       edgecolor="none" if (truth_val == 0 and p < t) else "black",
                       linewidth=0.4, zorder=2)

    # threshold line spanning both columns
    ax.axhline(t, color="black", linestyle="--", linewidth=1.5, zorder=3)
    ax.text(1.52, t, f"thr={t:.2f}", va="center", ha="left", fontsize=8)

    f1 = f1_score(truths, preds, zero_division=0)
    ax.set_title(f"{TOPIC_LABELS[i]}\nF1 = {f1:.2f}  (n_pos = {int(truths.sum())})",
                 fontsize=10, fontweight="bold")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Truth = 0\n(negative)", "Truth = 1\n(positive)"], fontsize=9)
    ax.set_xlim(-0.5, 1.9)
    ax.set_ylim(-0.02, 1.02)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.grid(True, linestyle=":", alpha=0.4)
    ax.set_axisbelow(True)
    if i % n_cols == 0:
        ax.set_ylabel("Predicted probability", fontweight="bold")

# hide any unused subplot cells
for j in range(n, len(axes)):
    axes[j].set_visible(False)

fig.legend(handles=[
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_TP, markersize=9, label="True Positive"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_FN, markersize=9, label="False Negative"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_FP, markersize=9, label="False Positive"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_TN, markersize=7, label="True Negative"),
    Line2D([0],[0], color="black", linestyle="--", label="Decision threshold"),
], loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.02))

fig.suptitle("Predicted Probability Distributions by Topic (test set)",
             fontsize=16, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()